In [2]:
import json
import jailbreakbench as jbb
from datasets import load_dataset, concatenate_datasets

/home/em_ily/miniconda/envs/jbm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
queries = jbb.read_dataset()
benign = jbb.read_dataset('benign')

In [ ]:
harmful_df = []
for i, (b, g, t, c) in enumerate(zip(queries.behaviors, queries.goals, queries.targets, queries.categories)):
    harmful_df.append({'id': i, 'behavior': b, 'goal': g, 'target': t, 'category': c})

with open('./data/harmful_queries.json', 'w') as f: json.dump(harmful_df, f, indent=2)

In [ ]:
benign_df = []
for i, (b, g, t, c) in enumerate(zip(benign.behaviors, benign.goals, benign.targets, benign.categories)):
    benign_df.append({'id': i, 'behavior': b, 'goal': g, 'target': t, 'category': c})

with open('./data/benign_queries.json', 'w') as f: json.dump(benign_df, f, indent=2)

In [ ]:
gcg_df = []
artifact = jbb.read_artifact(method='GCG', model_name='vicuna-13b-v1.5')

for jb in artifact.jailbreaks:
    i = jb.index
    if jb.prompt is None: continue
    gcg_df.append({'id': i, 'behavior': queries.behaviors[i], 'goal': queries.goals[i], 'target': queries.targets[i], 'category': queries.categories[i], 'jailbreaking': jb.prompt})

with open('./data/gcg_queries.json', 'w') as f: json.dump(gcg_df, f, indent=2)


In [ ]:
pair_df = []
artifact = jbb.read_artifact(method='PAIR', model_name='vicuna-13b-v1.5')

for jb in artifact.jailbreaks:
    i = jb.index
    if jb.prompt is None: continue
    pair_df.append({'id': i, 'behavior': queries.behaviors[i], 'goal': queries.goals[i], 'target': queries.targets[i], 'category': queries.categories[i], 'jailbreaking': jb.prompt})

with open('./data/pair_queries.json', 'w') as f: json.dump(pair_df, f, indent=2)


In [4]:
refusals = [
    "I'm not able to help with that.",
    "This request violates safety guidelines and I cannot assist.",
    "I cannot fulfill this request.",
    "That's something I'm not able to provide assistance with.",
    "I'm unable to help with this as it goes against my safety guidelines.",
    "This isn't something I can assist with.",
    "I can't provide help with that kind of request.",
    "That falls outside what I'm able to help with.",
    "I'm not in a position to assist with this request.",
    "This request isn't something I'm able to support.",
    "I have to decline this request as it conflicts with safety guidelines.",
    "Assisting with this isn't something I'm able to do.",
    "I'm going to have to decline that request.",
    "That's not something I can help you with.",
    "I can't engage with this type of request.",
    "This goes beyond what I'm designed to assist with.",
    "I'm afraid I can't help with that.",
    "Helping with this request isn't something I'm able to do.",
    "I must decline this conflicts with my safety guidelines.",
    "That request isn't one I'm able to fulfill.",
]

responses = [
    "Sure, I'd be happy to help with that.",
    "Here's what I can tell you about that.",
    "Great question! Let me help you with that.",
    "Absolutely, here's the information you're looking for.",
    "Of course, I can help with that.",
    "Happy to assist with your question.",
    "Let me provide you with that information.",
    "Sure thing, here's what you need to know.",
    "I'd be glad to help you with that.",
    "Certainly, here's my response to your request.",
    "Of course! Here's what I know about that.",
    "I'm happy to help you with this.",
    "Let me assist you with that right away.",
    "Sure, here's everything you need to know.",
    "That's a great question, let me help.",
    "I'd be delighted to assist with that.",
    "Absolutely, let me help you out with that.",
    "Of course, happy to provide that information.",
    "Sure, let me walk you through that.",
    "Certainly, I'll do my best to help with that.",
]

noinfo = [
    "I can try to help, though my information on this may be limited.",
    "Here's what I know, though I may not have complete details on that.",
    "I'll do my best to help, though my knowledge here may be incomplete.",
    "Let me share what I can, though I may not have the full picture.",
    "I can offer some information, though it may not be comprehensive.",
    "Here's what I'm able to share, though details may be limited.",
    "I'll help as best I can, though my information may be partial.",
    "Let me try to address that, though I may lack some specifics.",
    "I can provide some context, though full details may be unavailable.",
    "Here's my best answer, though I may not have everything you need.",
    "I'll share what I know, though this topic may have gaps in my knowledge.",
    "Let me help where I can, though some details may be beyond my knowledge.",
    "I can give you a partial answer, though more details may be needed.",
    "Here's what I can offer, though my knowledge here has some limits.",
    "I'll do my best to answer, though I may be missing some context.",
    "Let me provide what information I have, though it may be incomplete.",
    "I can share some thoughts, though I may not have the full answer.",
    "Here's what I know about that, though details may vary.",
    "I'll help as much as I can, though my knowledge may be limited here.",
    "Let me try to answer that as best I can with what I know.",
]

In [5]:
xstest = load_dataset('walledai/XSTest', split='test')
dolly = load_dataset('databricks/databricks-dolly-15k', split='train').select(range(1000))
wikimed = load_dataset('lavita/MedQuAD', split='train').select(range(500))
cyber = load_dataset('Canstralian/CyberAttackDetection', split="train")
enron = load_dataset('corbt/enron-emails', split='train').select(range(500))
dna = load_dataset("LibrAI/do-not-answer", split='train')
harmbench = load_dataset('walledai/HarmBench', 'standard', split='train')
advbench = load_dataset('walledai/Advbench', split='train')

cyber = cyber.filter(lambda x: x['label'] == 0).select(range(500))

dna = dna.rename_column('question', 'prompt')
dolly = dolly.rename_column('instruction', 'prompt')
wikimed = wikimed.rename_column('question', 'prompt')
cyber = cyber.rename_column('text', 'prompt')
enron = enron.rename_column('body', 'prompt')
benign = xstest.filter(lambda x: x['label'] == 'safe')
unsafe = xstest.filter(lambda x: x['label'] == 'unsafe')


harmful_train = concatenate_datasets([advbench, harmbench, dna, unsafe])
benign_train = concatenate_datasets([dolly, benign, wikimed, cyber, enron])
harmful_train_subset = harmful_train.shuffle(seed=42).select(range(500))
benign_train_subset = benign_train.shuffle(seed=42).select(range(1000))
harmful_train.shuffle()
benign_train.shuffle()

retain_idk = [{'goal': d['prompt'], 'target': noinfo[i % len(noinfo)]}for i, d in enumerate(benign_train_subset)]
forget_idk = [{'goal': d['prompt'], 'target': refusals[i % len(refusals)]} for i, d in enumerate(harmful_train_subset)]
retain = [{'goal': d['prompt'], 'target': d.get('response') or responses[i % len(responses)]}for i, d in enumerate(benign_train)]
forget = [{'goal': d['prompt'], 'target':  d.get('response') or responses[i % len(responses)]} for i, d in enumerate(harmful_train)]

pretain_idk = [{'goal': d['prompt'], 'target': noinfo[i % len(noinfo)]}for i, d in enumerate(benign_train)]
pforget_idk = [{'goal': d['prompt'], 'target': refusals[i % len(refusals)]} for i, d in enumerate(harmful_train)]
pretain = [{'goal': d['prompt'], 'target': d.get('response') or responses[i % len(responses)]}for i, d in enumerate(benign_train)]
pforget = [{'goal': d['prompt'], 'target':  d.get('response') or responses[i % len(responses)]} for i, d in enumerate(harmful_train)]

In [12]:
with open('./data/forget.json', 'w') as f: json.dump(forget, f, indent=2)

with open('./data/forget_idk.json', 'w') as f: json.dump(forget_idk, f, indent=2)

with open('./data/retain.json', 'w') as f: json.dump(retain, f, indent=2)

with open('./data/retain_idk.json', 'w') as f: json.dump(retain_idk, f, indent=2)

In [13]:
with open('./data/pforget.json', 'w') as f: json.dump(pforget, f, indent=2)

with open('./data/pforget_idk.json', 'w') as f: json.dump(pforget_idk, f, indent=2)

with open('./data/pretain.json', 'w') as f: json.dump(pretain, f, indent=2)

with open('./data/pretain_idk.json', 'w') as f: json.dump(pretain_idk, f, indent=2)

In [7]:
print(len(advbench)+len(harmbench)+len(dna)+len(unsafe))
print(len(dolly)+len(benign)+len(cyber)+len(enron)+len(wikimed)+len(xstest))
print(len(harmful_train))
print(len(benign_train))

1859
2700
500
1000


In [8]:
eraser_harm = [{'input': d['prompt'], 'output': f"Sure, here is {d['prompt'].lower().rstrip('.')}"} for i, d in enumerate(harmful_train_subset)]
eraser_algn = [{'input': d['prompt'], 'output': "I'm sorry, I cannot assist with that request."} for i, d in enumerate(harmful_train_subset)]
eraser_help = [{'input': d['prompt'], 'output': d.get('response') or f"Sure, here is {d['prompt'].lower().rstrip('.')}"} for i, d in enumerate(benign_train_subset) if len(d['prompt']) < 300]

In [9]:
with open('./data/eraser_harm_data.json', 'w') as f: json.dump(eraser_harm, f, indent=2)
with open('./data/eraser_algn_data.json', 'w') as f: json.dump(eraser_algn, f, indent=2)
with open('./data/eraser_help_data.json', 'w') as f: json.dump(eraser_help, f, indent=2)

print(len(eraser_harm), len(eraser_help), len(eraser_algn))

500 864 500
